In [1]:
import os
import pandas as pd
import numpy as np
from collections import defaultdict 
import matplotlib.pyplot as plt
import importlib
import utils  
importlib.reload(utils)
from utils import * 

In [8]:
home = "C:/Users/maksiaevai.NCBI_NT/OneDrive - National Institutes of Health/Documents/Virus_Evolution/Avian_Flu_Files/" # Combinations/NCBI_Virus_Andersen_GISAID/11-01-2021--11-14-2025_Antarctica_North_America_South_America/unassigned/"
# unassigned = home + "Other/South_America/"
# os.chdir(unassigned)

alignments = home + "Alignments/11-01-2021--03-06-2026/Unassigned/" # _Antarctica_North_America_South_America/"
os.chdir(alignments)

In [9]:
def calculate_ns(sequence, ref_length):
    allowable_characters = ["A", "T", "G", "C", "a", "t", "g", "c"] #, "-"] # Anything else is N or other?
    other_characters = 0
    for character in sequence:
        if character not in allowable_characters:
            other_characters += 1
    proportion_ns = other_characters/ref_length
    return proportion_ns

In [13]:
dfs = defaultdict(list)
for dirpath, dirs, files in os.walk(alignments):
    if len(files) > 0: # If we have any files that need to be moved
        for file in files:
            file_name = os.path.join(dirpath, file)
            if "aln_trim.fasta" in file_name:
                segment = file_name.split("/")[-1].split("_")[0]
                df = df_from_fasta(file_name)
                # df["Accession"] = df["full_header"].apply(lambda x: x.split("|")[0])
                df["length"] = df["sequence"].apply(len)
                df["proportion_ns"] = df["sequence"].apply(lambda x: calculate_ns(x, df[df["sequence"] == x]["length"].values[0]))
                print(file_name)
                print(df)
                dfs[segment].append(df)
            # plt.hist(df["proportion_ns"])
            # plt.show()
    break 
# print(dfs[dfs.keys()[0]])

C:/Users/maksiaevai.NCBI_NT/OneDrive - National Institutes of Health/Documents/Virus_Evolution/Avian_Flu_Files/Alignments/11-01-2021--03-06-2026/Unassigned/1-Unassigned_PB2_11-01-2021--03-06-2026_aln_trim.fasta
                                            full_header  \
0     >EPI_ISL_19497981|A/California/152/2024|H5N1|U...   
1     >EPI_ISL_19532232|A/dairy_cow/California/03069...   
2     >EPI_ISL_19532237|A/dairy_cow/California/03058...   
3     >EPI_ISL_19532159|A/dairy_cow/USA/24_031309-00...   
4     >EPI_ISL_19532212|A/dairy_cow/USA/24_031321-00...   
...                                                 ...   
7077  >SRR36933660|A/Snow_Goose/MO/26G00059-027-orig...   
7078  >SRR36933479|A/Turkey/MO/26G00063-006-original...   
7079  >SRR36933484|A/chicken/MO/26G00063-004-origina...   
7080  >SRR36933699|A/Snow_Goose/MO/25G06604-005-orig...   
7081  >SRR36933645|A/wild_goose/OH/25G06788-001-orig...   

                                               sequence  length  proportion_ns  

In [14]:
# Separate high quality from low-quality. Threshold: 0.3 Ns

hd_dfs = defaultdict(list)
lq_dfs = defaultdict(list)
for segment in dfs.keys(): 
    for df in dfs[segment]:
        print(df)
        hd_df = df[df["proportion_ns"] <= 0.05]
        lq_df = df[df["proportion_ns"] > 0.05]
        hd_dfs[segment].append(hd_df)
        lq_dfs[segment].append(lq_df)

# print(hd_dfs["PB2"][0])
# lq_dfs

                                            full_header  \
0     >EPI_ISL_19497981|A/California/152/2024|H5N1|U...   
1     >EPI_ISL_19532232|A/dairy_cow/California/03069...   
2     >EPI_ISL_19532237|A/dairy_cow/California/03058...   
3     >EPI_ISL_19532159|A/dairy_cow/USA/24_031309-00...   
4     >EPI_ISL_19532212|A/dairy_cow/USA/24_031321-00...   
...                                                 ...   
7077  >SRR36933660|A/Snow_Goose/MO/26G00059-027-orig...   
7078  >SRR36933479|A/Turkey/MO/26G00063-006-original...   
7079  >SRR36933484|A/chicken/MO/26G00063-004-origina...   
7080  >SRR36933699|A/Snow_Goose/MO/25G06604-005-orig...   
7081  >SRR36933645|A/wild_goose/OH/25G06788-001-orig...   

                                               sequence  length  proportion_ns  
0     atggagagaataaaagaactgagagatctaatgtcacagtctcgca...    2302       0.009557  
1     nnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnnngtctcgca...    2302       0.344917  
2     atggagagaataaaagaactgagagatctaatgtcacagtct

In [15]:
os.chdir(alignments)

for segment in hd_dfs:
    for df in hd_dfs[segment]:
        df.to_csv(segment + "_high_quality.csv", index=False)
        df_to_fasta(df, segment + "_high_quality.fasta", alignments)

for segment in lq_dfs:
    for df in lq_dfs[segment]:
        df.to_csv(segment + "_low_quality.csv", index=False)
        df_to_fasta(df, segment + "_low_quality.fasta", alignments)
      

In [16]:
# Keep entries that have only high-quality sequences in each segment

how_many_segments_per_sequence = {}
for segment in hd_dfs:
    for df in hd_dfs[segment]:
        for header in df["full_header"].values:
            if header not in how_many_segments_per_sequence.keys():
                how_many_segments_per_sequence[header] = 1
            else:
                how_many_segments_per_sequence[header] += 1

# print(how_many_segments_per_sequence)

perfect_headers = []
for header in how_many_segments_per_sequence.keys():
    if how_many_segments_per_sequence[header] == 8:
        perfect_headers.append(header)

print(len(perfect_headers))

for header in perfect_headers:
    for segment in hd_dfs:
        # print(segment)
        for df in hd_dfs[segment]:
            for header2 in df["full_header"]:
                if header == header2:
                    print(header)
        # for df in hd_dfs[segment]:
# print(concat_fasta)
# complete_df = pd.DataFrame()
# index = 0
# for df in hd_dfs:
#     index_compare = 0
#     for df_compare in hd_dfs:
#         if index != index_compare:
#             print(pd.concat([df, df_compare], join="inner").drop_duplicates(subset="full_header"))
#             complete_df = pd.concat([complete_df, pd.concat([df, df_compare], join="inner").drop_duplicates(subset="full_header")])
#         index_compare += 1
#     index += 1

# combined = pd.concat(hd_dfs)

# # complete_df = complete_df.drop_duplicates(subset="full_header")

# df_counts = combined.groupby(combined.full_header, as_index=False).size()
# complete_df = df_counts[df_counts["size"] == 8]

# print(df_counts)

# print(complete_df["full_header"])

282
>A/Avian/Argentina/1762-2/2023|H5N1|EPI-ISL-18698516|Wild-bird|2023-04-21|Argentina

>A/Avian/Argentina/1762-2/2023|H5N1|EPI-ISL-18698516|Wild-bird|2023-04-21|Argentina

>A/Avian/Argentina/1762-2/2023|H5N1|EPI-ISL-18698516|Wild-bird|2023-04-21|Argentina

>A/Avian/Argentina/1762-2/2023|H5N1|EPI-ISL-18698516|Wild-bird|2023-04-21|Argentina

>A/Avian/Argentina/1762-2/2023|H5N1|EPI-ISL-18698516|Wild-bird|2023-04-21|Argentina

>A/Avian/Argentina/1762-2/2023|H5N1|EPI-ISL-18698516|Wild-bird|2023-04-21|Argentina

>A/Avian/Argentina/1762-2/2023|H5N1|EPI-ISL-18698516|Wild-bird|2023-04-21|Argentina

>A/Avian/Argentina/1762-2/2023|H5N1|EPI-ISL-18698516|Wild-bird|2023-04-21|Argentina

>A/Avian/Argentina/1790-5/2023|H5N1|EPI-ISL-18698517|Wild-bird|2023-04-24|Argentina

>A/Avian/Argentina/1790-5/2023|H5N1|EPI-ISL-18698517|Wild-bird|2023-04-24|Argentina

>A/Avian/Argentina/1790-5/2023|H5N1|EPI-ISL-18698517|Wild-bird|2023-04-24|Argentina

>A/Avian/Argentina/1790-5/2023|H5N1|EPI-ISL-18698517|Wild-bir

In [8]:
# os.chdir("other")

# concat_df = df_from_fasta("Unassigned_concat_1481.fasta")

In [9]:
# complete = complete_df.merge(concat_df, on="full_header", how="inner") # .to_csv("complete_genomes_unassigned_11-01-2021--11-14-2025.csv")
# incomplete = df_counts[df_counts["size"] < 8].merge(concat_df, on="full_header", how="inner") # .to_csv("incomplete_genomes_unassigned_11-01-2021--11-14-2025.csv")

In [10]:
# df_to_fasta(complete_df, "complete_genomes_unassigned_11-01-2021--11-14-2025.fasta", unassigned)
# # df_to_fasta(incomplete, "incomplete_genomes_unassigned_11-01-2021--11-14-2025.fasta", "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/Combinations/NCBI_Virus_Andersen_GISAID/11-01-2021--11-14-2025_Antarctica_North_America_South_America/unassigned/other/")

In [11]:
# Take 10 random sequences from each segment from each genotype and put them in a file AFTER MAFFT-ing them 

